In [ ]:
from __future__ import annotations

import concurrent.futures
import dataclasses
import itertools
from collections.abc import Callable, Sequence
import multiprocessing
from typing import Any, Final, Protocol, final, overload, override
import concurrent.futures

import numpy as np
import numpy.typing as npt

np.set_printoptions(threshold=10_000)
np.set_printoptions(linewidth=10_000)


# ==================================================================================================
# Controller
#
# V             - value         - O(T)      ∈ [0; +∞)
# D             - debt          - O(exp(T)) ∈ [0; +∞)
# d = ln(D + 1) - remapped debt - O(T)      ∈ [0; +∞)


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class GridParams:
    D_min: float
    D_max: float
    nD_int: int

    T0: float
    T1: float

    CFL: float
    Δt_min: float
    Δt_max: float


@final
class StateLayer:
    __slots__: Final = ("__x0", "__x1", "__xs", "__iΔx")

    def __init__(self, x0: float, x1: float, n_int: int) -> None:
        assert x1 > x0
        self.__x0: Final = x0
        self.__x1: Final = x1
        self.__xs: Final = np.zeros(n_int + 2, dtype=np.float32)
        self.__iΔx: Final = (n_int + 1) / (x1 - x0)

    @property
    def x0(self) -> float:
        return self.__x0

    @property
    def x1(self) -> float:
        return self.__x1

    @property
    def iΔx(self) -> float:
        return self.__iΔx

    @property
    def w(self) -> npt.NDArray[np.float32]:
        return self.__xs

    def lookup(self, x_vals: npt.NDArray[np.float32]) -> npt.NDArray[np.float32]:
        i: Final = (x_vals - self.__x0) * self.__iΔx

        # Clamp indices for interpolation/extrapolation
        iM = np.clip(np.floor(i).astype(int), 0, len(self.__xs) - 2)
        iP = np.minimum(iM + 1, len(self.__xs) - 1)

        # Linear interpolation
        α = i - iM
        result = (1 - α) * self.__xs[iM] + α * self.__xs[iP]

        # Handle out-of-bounds (below x0)
        result = np.where(i < 0, -np.inf, result)

        return result


class Model(Protocol):
    def dotV(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]: ...
    def dotD(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]: ...


@final
class DSpace:
    def __init__(self, D_min: float, D_max: float, n_int: int) -> None:
        self.__d0: Final = self.D_to_d(D_min)
        self.__d1: Final = self.D_to_d(D_max)
        self.__Δd: Final = (self.__d1 - self.__d0) / (n_int + 1)
        self.__n_int: Final = n_int

    @property
    def d0(self) -> float:
        return self.__d0

    @property
    def d1(self) -> float:
        return self.__d1

    @property
    def Δd(self) -> float:
        return self.__Δd

    def D_to_k(self, D: float, *, clamp: bool) -> int:
        return self.d_to_k(self.D_to_d(D), clamp=clamp)

    def d_to_k(self, d: float, *, clamp: bool) -> int:
        k: Final = int(d / self.__Δd)
        if clamp:
            return min(max(0, k), self.__n_int + 1)
        else:
            return k

    def k_to_D(self, k: int) -> float:
        return self.d_to_D(self.k_to_d(k))

    def k_to_d(self, k: int) -> float:
        return self.d0 + k * self.Δd

    def dotD_to_dotd(self, dotD: npt.NDArray[np.float32], D: npt.NDArray[np.float32]) -> npt.NDArray[np.float32]:
        return dotD / (1 + D)

    @overload
    def D_to_d(self, D: float) -> float: ...

    @overload
    def D_to_d(self, D: npt.NDArray[np.float32]) -> npt.NDArray[np.float32]: ...

    def D_to_d(self, D: Any) -> Any:
        return np.log(1 + D)

    @overload
    def d_to_D(self, d: float) -> float: ...

    @overload
    def d_to_D(self, d: npt.NDArray[np.float32]) -> npt.NDArray[np.float32]: ...

    def d_to_D(self, d: Any) -> Any:
        return np.exp(d) - 1


@final
class AdaptiveStep:
    def __init__(self, cfl: float, Δt_min: float, Δt_max: float) -> None:
        self.__max_dot: float = -np.inf
        self.__Δt_min: Final = Δt_min
        self.__Δt_max: Final = Δt_max
        self.__cfl: Final = cfl
        self.__stable = True

    @property
    def stable(self) -> bool:
        return self.__stable

    def reset(self) -> None:
        self.__max_dot = -np.inf

    def consume_dotx(self, dotx: float) -> None:
        self.__max_dot = max(self.__max_dot, dotx)

    def compute_Δt(self, Δx: float) -> float:
        Δt: Final = self.__cfl * Δx / self.__max_dot
        if Δt >= self.__Δt_min:
            return min(Δt, self.__Δt_max)
        self.__stable = False
        return self.__Δt_min


@final
class Controller:
    __slots__: Final = ("__gp", "__model", "__ts", "__bs", "__stable", "__dsp")

    def __init__(self, model: Model, gp: GridParams) -> None:
        ts: Final = list[float]()
        bs: Final = list[npt.NDArray[np.bool]]()

        dsp: Final = DSpace(gp.D_min, gp.D_max, gp.nD_int)
        Δt_step: Final = AdaptiveStep(gp.CFL, gp.Δt_min, gp.Δt_max)

        # NOTE: W(:, T) = 0
        curr_ws = StateLayer(dsp.d0, dsp.d1, gp.nD_int)
        next_ws = StateLayer(dsp.d0, dsp.d1, gp.nD_int)

        # Pre-compute the spatial grid
        kds: Final = np.arange(0, gp.nD_int + 2, dtype=np.int32)
        d_grid: Final = dsp.d0 + kds * dsp.Δd
        D_grid: Final = dsp.d_to_D(d_grid)

        def compute_dotd(D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
            return dsp.dotD_to_dotd(model.dotD(D, b), D)

        def compute_dotv(D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
            return model.dotV(D, b)

        def compute_w_plus(
            d: npt.NDArray[np.float32], D: npt.NDArray[np.float32], b: bool, Δt: float
        ) -> npt.NDArray[np.float32]:
            dotd: Final = compute_dotd(D, b)
            dotV: Final = compute_dotv(D, b)
            Δt_step.consume_dotx(float(np.max(np.abs(dotd))))

            next_d: Final = d + dotd * np.float32(Δt)
            next_w: Final = next_ws.lookup(next_d) + dotV * np.float32(Δt)  # type: ignore
            return next_w  # type: ignore

        # Compute first adaptive time step
        dotd_0 = compute_dotd(D_grid, False)
        dotd_1 = compute_dotd(D_grid, True)

        max_dotd = max(np.max(np.abs(dotd_0)), np.max(np.abs(dotd_1)))
        Δt_step.consume_dotx(float(max_dotd))
        Δt = Δt_step.compute_Δt(dsp.Δd)

        ts.append(gp.T1)
        bs.append(np.ones_like(curr_ws.w, dtype=np.bool))

        t = gp.T1 - Δt

        next_round = True
        while next_round:
            # compute a current layer time
            t -= Δt

            next_round = t > gp.T0
            if not next_round:
                t = gp.T0

            # reset layer params
            Δt_step.reset()

            # layer-local params
            curr_bs = np.zeros_like(curr_ws.w, dtype=np.bool)

            # Determine optimal control (kd=0 -> b=True from stability purposes)
            ws_0 = compute_w_plus(d_grid[1:], D_grid[1:], False, Δt)
            ws_1 = compute_w_plus(d_grid[0:], D_grid[0:], True, Δt)

            curr_bs[0] = True
            curr_ws.w[0] = ws_1[0]

            curr_bs[1:] = ws_1[1:] >= ws_0
            curr_ws.w[1:] = np.where(curr_bs[1:], ws_1[1:], ws_0)

            # save results
            ts.append(t)
            bs.append(curr_bs.copy())

            # prepare next round
            curr_ws, next_ws = next_ws, curr_ws

            Δt = Δt_step.compute_Δt(dsp.Δd)

        # prepare forward propagation
        ts.reverse()
        bs.reverse()

        self.__ts: Final = ts
        self.__bs: Final = bs
        self.__dsp: Final = dsp
        self.__stable: Final = Δt_step.stable

        self.__gp: Final = gp
        self.__model: Final = model

    @property
    def ts(self) -> Sequence[float]:
        return self.__ts

    @property
    def stable(self) -> bool:
        return self.__stable

    def b(self, kt: int, D: float) -> bool:
        return self.__bs[kt][self.__dsp.D_to_k(D, clamp=True)]

    def bs(self) -> npt.NDArray[np.bool]:
        return np.array(self.__bs, dtype=np.bool)

    def Ds(self) -> npt.NDArray[np.float32]:
        return np.array([self.__dsp.k_to_D(k) for k in range(self.__gp.nD_int + 2)])

    def grid(self) -> GridParams:
        return self.__gp

    def model(self) -> Model:
        return self.__model


# ==================================================================================================
# Simulation


@final
@dataclasses.dataclass(kw_only=True, slots=True, frozen=True)
class ModelParams:
    α: float
    μ: float
    r: float


@final
@dataclasses.dataclass(kw_only=True, slots=True, frozen=True)
class ModelParamsSpace:
    α: npt.NDArray[np.float32]
    μ: npt.NDArray[np.float32]
    r: npt.NDArray[np.float32]


@final
@dataclasses.dataclass(kw_only=True, slots=True, frozen=True)
class InitialParams:
    V0: float
    D0: float


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SingleCurve:
    b_opt: npt.NDArray[np.bool]
    V_opt: npt.NDArray[np.float32]
    D_opt: npt.NDArray[np.float32]


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SimulationResults:
    ts: npt.NDArray[np.float32]
    Ds: npt.NDArray[np.float32]
    bs: npt.NDArray[np.bool]
    curves: dict[InitialParams, SingleCurve]


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SimulationBundle:
    results: dict[ModelParams, SimulationResults]


def simulate(controller: Controller, ips: Sequence[InitialParams]) -> SimulationResults:
    md: Final = controller.model()
    ts: Final = controller.ts

    def compute_curve(ip: InitialParams) -> SingleCurve:
        curve: Final = SingleCurve(
            b_opt=np.zeros(len(ts), dtype=np.bool),
            V_opt=np.zeros(len(ts), dtype=np.float32),
            D_opt=np.zeros(len(ts), dtype=np.float32),
        )
        curve.V_opt[0] = ip.V0
        curve.D_opt[0] = ip.D0

        for kt in range(len(ts)):
            V = curve.V_opt[kt]
            D = curve.D_opt[kt]

            b = controller.b(kt, D)
            curve.b_opt[kt] = b

            if kt + 1 < len(ts):
                Δt = ts[kt + 1] - ts[kt]
                curve.V_opt[kt + 1] = V + md.dotV(D, b) * Δt
                curve.D_opt[kt + 1] = max(D + md.dotD(D, b) * Δt, 0.0)

        return curve

    return SimulationResults(
        ts=np.array(ts, dtype=np.float32),
        Ds=controller.Ds(),
        bs=controller.bs(),
        curves={ip: compute_curve(ip) for ip in ips},
    )


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SimulationPack:
    params: ModelParams

    model_factory: Callable[[ModelParams], Model]
    space: ModelParamsSpace
    ips: Sequence[InitialParams]
    gp: GridParams


def simulate_cell(pack: SimulationPack) -> tuple[ModelParams, SimulationResults]:
    model = pack.model_factory(pack.params)
    controller = Controller(model, pack.gp)
    if not controller.stable:
        print(f"WARNING: unstable solution for: {pack.params}\n", end="")
    return (pack.params, simulate(controller, pack.ips))


def simulate_space(
    model_factory: Callable[[ModelParams], Model],
    space: ModelParamsSpace,
    ips: Sequence[InitialParams],
    gp: GridParams,
) -> SimulationBundle:
    packs: Final = [
        SimulationPack(
            params=ModelParams(α=α, μ=μ, r=r),
            model_factory=model_factory,
            space=space,
            ips=ips,
            gp=gp,
        )
        for α, μ, r in itertools.product(space.α, space.μ, space.r)
    ]

    with concurrent.futures.ProcessPoolExecutor(mp_context=multiprocessing.get_context("fork")) as pool:
        results_dict = dict(pool.map(simulate_cell, packs, chunksize=10))
    return SimulationBundle(results=results_dict)


# ==================================================================================================


@final
class ModelExp(Model):
    __slots__: Final = "__mp"

    def __init__(self, mp: ModelParams) -> None:
        self.__mp: Final = mp

    @override
    def dotV(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
        mp: Final = self.__mp
        return int(b) * np.exp(-mp.μ * D)

    @override
    def dotD(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
        mp: Final = self.__mp
        return (((mp.α + 1) * int(b) - 1) * np.exp(-mp.μ * D) + mp.r * D).astype(np.float32, copy=False)


@final
class ModelHyper(Model):
    __slots__: Final = "__mp"

    def __init__(self, mp: ModelParams) -> None:
        self.__mp: Final = mp

    @override
    def dotV(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
        mp: Final = self.__mp
        return (int(b) / (1 + mp.μ * D)).astype(np.float32, copy=False)

    @override
    def dotD(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
        mp: Final = self.__mp
        return (((mp.α + 1) * int(b) - 1) / (1 + mp.μ * D) + mp.r * D).astype(np.float32, copy=False)

In [ ]:
BASE = 20

packs = ModelParamsSpace(
    α=np.linspace(0, 2.00, num=BASE, dtype=np.float32),
    μ=np.linspace(0, 10.0, num=BASE, dtype=np.float32),
    r=np.linspace(0, 1.00, num=BASE, dtype=np.float32),
)
initial_params = [InitialParams(V0=0.0, D0=D0) for D0 in np.linspace(0, 0.5, 40)]
grid_params = GridParams(
    D_min=0,
    D_max=1.5,
    nD_int=100,
    T0=0,
    T1=1,
    CFL=0.9,
    Δt_min=0.0001,
    Δt_max=0.1000,
)

bundle = simulate_space(ModelExp, packs, initial_params, grid_params)
print(max(max(curve.D_opt) for x in bundle.results.values() for curve in x.curves.values()))
print(max(max(curve.V_opt) for x in bundle.results.values() for curve in x.curves.values()))

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
import plotly.subplots

key, data = list(bundle.results.items())[np.sum(int(BASE * 0.78) * BASE ** np.array([2, 1, 0]))]
assert isinstance(key, ModelParams)
assert isinstance(data, SimulationResults)

# Create figure with subplots
fig: go.Figure = plotly.subplots.make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f"Optimal control b<sub>p</sub><sup>q</sup>, α={key.α:.2f}, μ={key.μ:.2f}, r={key.r:.2f}",
        f"Phase portrait, α={key.α:.2f}, μ={key.μ:.2f}, r={key.r:.2f}",
    ),
    horizontal_spacing=0.12,
    shared_xaxes=True,
)

# Left subplot: D-T trajectories with contour
fig.add_trace(
    go.Contour(
        z=data.bs.astype(np.int32),
        x=data.Ds,
        y=data.ts,
        colorscale=[[0, "blue"], [1, "rgb(70, 230, 30)"]],
        showscale=True,
    ),
    row=1,
    col=1,
)

for ti in data.curves.values():
    fig.add_trace(
        go.Scatter(x=ti.D_opt, y=data.ts, mode="lines", line=dict(color="red", width=2), showlegend=False),
        row=1,
        col=1,
    )

# Right subplot: Phase portraits
for ti in data.curves.values():
    fig.add_trace(
        go.Scatter(x=ti.D_opt, y=ti.V_opt, mode="lines", line=dict(width=2), showlegend=False),
        row=1,
        col=2,
    )

# Update axes labels
fig.update_xaxes(title_text="D", row=1, col=1)
fig.update_yaxes(title_text="t", row=1, col=1)
fig.update_xaxes(title_text="D", row=1, col=2)
fig.update_yaxes(title_text="V", row=1, col=2)

fig.update_xaxes(range=[data.Ds.min(), data.Ds.max()], row=1, col=1)
fig.update_yaxes(range=[data.ts.min(), data.ts.max()], row=1, col=1)
fig.update_xaxes(range=[data.Ds.min(), data.Ds.max()], row=1, col=2)

# Update layout
fig.update_layout(
    width=1200,
    height=600,
    showlegend=False,
    margin=dict(l=50, r=50, t=80, b=50),
)

fig.show()

In [ ]:
# D0 -> V1

fig = go.Figure()

for idx in [np.sum(x * BASE ** np.array([2, 1, 0])) for x in range(0, BASE, 1)]:
    key, data = list(bundle.results.items())[idx]

    fig.add_trace(
        go.Scatter(
            x=[curve.D_opt[0] for curve in data.curves.values()],
            y=[curve.V_opt[-1] for curve in data.curves.values()],
            mode="lines+markers",
            name=f"α={key.α:.2f}, μ={key.μ:.2f}, r={key.r:.2f}",
        )
    )

fig.update_layout(
    title="Relation between the resulting value V(T) and an initial system debt D(0)",
    xaxis_title="D(0)",
    yaxis_title="V(T)",
    width=800,
    height=800,
    showlegend=True,
)
fig.show()

In [ ]:
import plotly.graph_objects as go
import scipy.interpolate

# Switch points

xs = []
ys = []
for key, data in bundle.results.items():
    switch_points = [
        data.ts[len(curve.b_opt) - 1 - curve.b_opt[::-1].argmin()]
        for curve in data.curves.values()
        if curve.b_opt.min() < 0.5
    ]

    for ti in switch_points:
        xs.append(key.μ)
        ys.append(ti)

xs = np.array(xs)
ys = np.array(ys)

# Calculate the average t_rush for each unique μ value
unique_μs = np.sort(np.unique(xs))
μ_smooth = np.linspace(unique_μs.min(), unique_μs.max(), 200)

# Print the plot
fig = go.Figure()
fig.add_trace(go.Histogram2d(x=xs, y=ys, nbinsx=BASE // 2, nbinsy=BASE // 2, histnorm="probability"))

for q in [50]:
    t_avg = np.array([np.quantile(ys[xs == μ], q / 100) for μ in unique_μs])
    t_smooth = scipy.interpolate.interp1d(unique_μs, t_avg, kind="cubic")(μ_smooth)
    fig.add_trace(
        go.Scatter(
            x=μ_smooth,
            y=t_smooth,
            mode="lines",
            line={"color": "white", "width": 4, "dash": "dot"},
            showlegend=False,
        )
    )

fig.update_layout(
    title_text="t<sub>rush</sub> = max(t) | b(τ)=1 for any τ > t",
    title_x=0.5,
    title_xanchor="center",
    title_yanchor="top",
    xaxis_title="μ",
    autosize=False,
    width=800,
    height=800,
    margin={"l": 65, "r": 50, "b": 65, "t": 90},
)
fig.show()